# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
95,Adelie,4300.0,40.8,2.583141
297,Gentoo,6000.0,51.1,2.297785
157,Chinstrap,3950.0,45.2,1.933393
248,Gentoo,4600.0,48.2,1.979993
311,Gentoo,5400.0,52.2,1.981768
128,Adelie,3050.0,39.0,2.005260
2,Adelie,3250.0,40.3,2.001121
259,Gentoo,5350.0,48.7,2.255775
205,Chinstrap,4050.0,50.7,1.575575
74,Adelie,3700.0,35.5,2.935925


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
142,Adelie,3050.0,3.050,False
71,Adelie,3900.0,3.900,False
326,Gentoo,4700.0,4.700,True
316,Gentoo,4925.0,4.925,True
80,Adelie,3200.0,3.200,False
155,Chinstrap,3525.0,3.525,False
254,Gentoo,5150.0,5.150,True
211,Chinstrap,3525.0,3.525,False
253,Gentoo,6050.0,6.050,True
131,Adelie,3500.0,3.500,False


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
31,Adelie,3900.0,3.900,8.598018
324,Gentoo,4725.0,4.725,10.416829
19,Adelie,4200.0,4.200,9.259404
171,Chinstrap,4400.0,4.400,9.700328
38,Adelie,3300.0,3.300,7.275246
216,Chinstrap,3400.0,3.400,7.495708
330,Gentoo,5000.0,5.000,11.023100
310,Gentoo,4950.0,4.950,10.912869
71,Adelie,3900.0,3.900,8.598018
300,Gentoo,4625.0,4.625,10.196367


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
49,Adelie,True
84,Adelie,True
317,Gentoo,False
128,Adelie,True
192,Chinstrap,False
329,Gentoo,False
144,Adelie,True
111,Adelie,True
282,Gentoo,False
326,Gentoo,False


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
169,Chinstrap,3700.0,False
6,Adelie,3625.0,False
332,Gentoo,4650.0,True
74,Adelie,3700.0,False
18,Adelie,3325.0,False
91,Adelie,4300.0,True
32,Adelie,3300.0,False
189,Chinstrap,4800.0,True
191,Chinstrap,4500.0,True
243,Gentoo,5050.0,True


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
93,Adelie,4.450
168,Chinstrap,3.300
336,Gentoo,4.875
183,Chinstrap,4.300
194,Chinstrap,3.550
235,Gentoo,5.850
102,Adelie,3.075
224,Gentoo,5.400
67,Adelie,4.100
340,Gentoo,4.850


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
140,Adelie,Dream,3400.0,40.2,2.103908
328,Gentoo,Biscoe,4575.0,43.3,2.440143
309,Gentoo,Biscoe,5550.0,52.1,2.044643
260,Gentoo,Biscoe,3950.0,42.7,2.166413
343,Gentoo,Biscoe,5400.0,49.9,2.168666
243,Gentoo,Biscoe,5050.0,46.3,2.355751
334,Gentoo,Biscoe,4375.0,46.2,2.049718
223,Gentoo,Biscoe,5700.0,50.0,2.280000
325,Gentoo,Biscoe,5500.0,46.8,2.511140
317,Gentoo,Biscoe,4875.0,46.9,2.216302


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
7,Adelie,4675.0,heavy
132,Adelie,3500.0,light
279,Gentoo,5550.0,heavy
167,Chinstrap,4050.0,heavy
149,Adelie,3750.0,light
145,Adelie,3650.0,light
242,Gentoo,4400.0,heavy
277,Gentoo,5000.0,heavy
229,Gentoo,5150.0,heavy
273,Gentoo,5000.0,heavy


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
215,Chinstrap,4000.0,medium
316,Gentoo,4925.0,large
68,Adelie,3050.0,small
230,Gentoo,4650.0,large
273,Gentoo,5000.0,large
112,Adelie,3200.0,small
117,Adelie,3775.0,medium
184,Chinstrap,3350.0,small
179,Chinstrap,3800.0,medium
148,Adelie,3450.0,small


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
21,Adelie,Biscoe,BIS
90,Adelie,Dream,OTH
224,Gentoo,Biscoe,BIS
334,Gentoo,Biscoe,BIS
212,Chinstrap,Dream,OTH
283,Gentoo,Biscoe,BIS
42,Adelie,Dream,OTH
146,Adelie,Dream,OTH
188,Chinstrap,Dream,OTH
40,Adelie,Dream,OTH


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
87,Adelie,3.500,7.716170
299,Gentoo,5.950,13.117489
46,Adelie,3.425,7.550823
30,Adelie,3.250,7.165015
79,Adelie,4.000,8.818480


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — dictionary unpacking (`**{...}`)

Python keyword argument syntax requires valid identifier names (`key=value`), so bare names with spaces like `my col = "..."` cause a Python `SyntaxError`.

To create a new column whose name contains spaces, pass the column definition in a dictionary and unpack it using `**`:


In [18]:
# Create a new column with spaces using dictionary unpacking
(
    spaced
    .pt.mutate(**{'body mass ratio': '[body mass g] / [bill length mm]'})
)


,body mass g,bill length mm,body mass ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


In [19]:
# Define multiple new columns with spaces at once
(
    penguins
    .pt.mutate(**{
        'body mass kg': 'body_mass_g / 1000',
        'bill ratio': 'bill_length_mm / bill_depth_mm',
    })
    .pt.select('species', 'body mass kg', 'bill ratio')
    .head()
)


,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```
